# Extra — Frontier Models Generating SVGs

Compares how different frontier models handle a creative coding task: generating an SVG image (a much harder skill than image generation, since the model has to describe the picture in lines and shapes).

Uses OpenRouter as a single gateway to all the models.

## Setup

In [ ]:
import os
import time
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from revealer import reveal

load_dotenv(override=True)

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY and OPENROUTER_API_KEY.startswith("sk-or-"):
    print("OpenRouter API key looks good.")
else:
    print("OpenRouter API key not set or doesn't look right.")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

## The challenge

In [ ]:
challenge = "a panda rollerblading to work"
prompt = f"Generate an SVG of {challenge}. Respond with the SVG only, no code blocks."
messages = [{"role": "user", "content": prompt}]

## Helper: send the prompt to a model and time the call

In [ ]:
def artist(model, effort=None):
    try:
        start = datetime.now()
        response = openrouter.chat.completions.create(
            model=model, messages=messages, reasoning_effort=effort
        )
        result = response.choices[0].message.content
        elapsed = (datetime.now() - start).total_seconds()
        heading = f"### {model}\n**Time:** {elapsed // 60:.0f} min {elapsed % 60:.0f} s\n\n"
        return heading, result
    except Exception as e:
        print(f"Model {model} failed: {e}")
        return f"### {model}\n**Error:** {e}\n\n", None

## Run the challenge across multiple frontier models

In [ ]:
results = [
    artist("openai/gpt-oss-120b"),
    artist("openai/gpt-5-nano", effort="low"),
    artist("deepseek/deepseek-v3.2"),
    artist("moonshotai/kimi-k2-thinking"),
    artist("x-ai/grok-4.1-fast"),
    artist("anthropic/claude-opus-4.5"),
    artist("openai/gpt-5.2", effort="high"),
    artist("google/gemini-3-pro-preview")
]

## Reveal each SVG with a typewriter-style animation

In [ ]:
for result in results:
    try:
        display(Markdown(result[0]))
        reveal(result[1])
        time.sleep(12)
    except Exception as e:
        print(f"Error displaying result: {e}")